In [1]:
%pip install confluent-kafka

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import time
from itertools import islice
from confluent_kafka import Producer

MONTHS = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12",
}

def event_time_key(row):
    parts = row["DATE OCC"].split()
    yyyy, mmm, dd = parts[0], MONTHS.get(parts[1], "00"), parts[2]
    hhmm = row["TIME OCC"].zfill(4)
    return f"{yyyy}-{mmm}-{dd} {hhmm}"

max_rows = 20 # stopping at 20 rows for live demo, set to None to replay all rows (Note: total rows is a little more than a million, might take longer to run)
producer = Producer({"bootstrap.servers": "kafka:9092"})
topic = "crime-events"

with open("Crime Data.csv", mode="r", encoding="latin-1") as f:
    reader = csv.DictReader(f)
    rows = list(islice(reader, max_rows)) if max_rows else list(reader)

# Sort rows in ascending event-time order (DATE OCC + TIME OCC)
rows.sort(key=event_time_key)

for i, row in enumerate(rows, start=1):
    event = {
        "DR_NO": row["DR_NO"],
        "DATE OCC": row["DATE OCC"],
        "TIME OCC": row["TIME OCC"].zfill(4),
        "AREA NAME": row["AREA NAME"],
        "Crm Cd Desc": row["Crm Cd Desc"],
    }
    producer.produce(
        topic,
        key=row["AREA NAME"].encode("utf-8"),
        value=json.dumps(event).encode("utf-8")
    )
    print(f"Produced row {i}: DR_NO={event['DR_NO']} | DATE OCC={event['DATE OCC']} | TIME OCC={event['TIME OCC']} | AREA={event['AREA NAME']}")
    time.sleep(0.2)   # small delay to simulate live replay

producer.flush()
print("Producer finished replaying rows in event-time order.")

Produced row 1: DR_NO=201405023 | DATE OCC=2020 Jan 19 12:00:00 AM | TIME OCC=1850 | AREA=Pacific
Produced row 2: DR_NO=201405970 | DATE OCC=2020 Feb 01 12:00:00 AM | TIME OCC=1658 | AREA=Pacific
Produced row 3: DR_NO=201406733 | DATE OCC=2020 Feb 13 12:00:00 AM | TIME OCC=2300 | AREA=Pacific
Produced row 4: DR_NO=241708596 | DATE OCC=2020 Mar 02 12:00:00 AM | TIME OCC=1200 | AREA=Devonshire
Produced row 5: DR_NO=201707577 | DATE OCC=2020 Mar 20 12:00:00 AM | TIME OCC=1320 | AREA=Devonshire
Produced row 6: DR_NO=240317069 | DATE OCC=2020 Apr 16 12:00:00 AM | TIME OCC=1350 | AREA=Southwest
Produced row 7: DR_NO=240605846 | DATE OCC=2020 Jun 20 12:00:00 AM | TIME OCC=0001 | AREA=Hollywood
Produced row 8: DR_NO=201710725 | DATE OCC=2020 Jul 02 12:00:00 AM | TIME OCC=0500 | AREA=Devonshire
Produced row 9: DR_NO=201115217 | DATE OCC=2020 Jul 07 12:00:00 AM | TIME OCC=1400 | AREA=Northeast
Produced row 10: DR_NO=201514848 | DATE OCC=2020 Aug 17 12:00:00 AM | TIME OCC=0400 | AREA=N Hollywood
